In [1]:
from pyflink.table import (
    EnvironmentSettings,
    TableEnvironment
)

In [2]:
env_settings = (
    EnvironmentSettings.new_instance()
    .in_streaming_mode()
    .build()
)
t_env = TableEnvironment.create(env_settings)
conf = t_env.get_config().get_configuration()

/usr/local/lib/python3.11/dist-packages/apache_beam/runners/portability/stager.py:63: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
conf.set_string("execution.target", "remote")
conf.set_string("rest.address", "jobmanager")
conf.set_string("rest.port", "8081")
conf.set_string("parallelism.default", "1")

In [4]:
t_env.execute_sql("""
CREATE TABLE source_table (
    id INT,
    name STRING,
    age INT
) WITH (
    'connector' = 'datagen',
    'rows-per-second' = '2'
)
""")

In [4]:
t_env.execute_sql("""
CREATE TABLE sink_table (
    id INT,
    name STRING,
    age INT
) WITH (
    'connector' = 'filesystem',
    'path' = 'file:///workspace/output/json_sink',
    'format' = 'json'
)
""")

In [8]:
result = t_env.execute_sql("""
INSERT INTO sink_table
SELECT id, name, age FROM source_table
""")

In [13]:
result.get_job_client().cancel()

In [5]:
t_env.execute_sql("""
INSERT INTO sink_table
SELECT * FROM (VALUES (1, 'ABC', 24), (2, 'DEF', 25)) AS t(id, name, age)
""")

In [10]:
source_table = t_env.from_elements(
    [(1, 'ABC', 26), (2, 'DEF', 27)],
    ['id', 'name', 'age']
)
source_table.print_schema()

(
  `id` BIGINT,
  `name` STRING,
  `age` BIGINT
)


In [11]:
t_env.create_temporary_view("source_view4", source_table)

t_env.execute_sql("""
INSERT INTO sink_table
SELECT CAST(id AS INT), name, CAST(age AS INT) FROM source_view4
""")